# **Proyecto Etapa 2. Construcción de Muestra Representativa — Particionamiento**

### **Curso: TC5057 · Análisis de Grandes Volúmenes de Datos**
#### **Tecnológico de Monterrey**
##### **Profesor Titular: Dr. Iván Olmos Pineda**

---

### **Equipo #17**
#### **Tutor: José Carlos Soto**

| Nombre | Matrícula |
|--------|----------|
| Ana Bonavides Aguilar | A01423281 |
| Alejandro José Martínez Ubeda | A01797775 |
| José Andrés Orantes Guillén | A01174130 |
| Diego Falcón Costilla | A01139580 |

## Descripción del Dataset

El dataset procesado en este notebook es el **GTEx Analysis V10 — Gene Expression TPM**, proveniente del proyecto Genotype-Tissue Expression (GTEx) del NIH / Broad Institute.

| Campo | Detalle |
|-------|--------|
| **Archivo principal** | `GTEx_Analysis_2022-06-06_v10_RNASeQCv2.4.2_gene_tpm_non_lcm.gct` |
| **Dimensiones** | 59,033 genes × 19,616 muestras de tejido humano |
| **Formato** | GCT 1.2 (TSV con 2 filas de metadatos al inicio) |
| **Metadatos de muestras** | `GTEx_Analysis_v10_Annotations_SampleAttributesDS.txt` (48,231 filas) |
| **Metadatos de donantes** | `GTEx_Analysis_v10_Annotations_SubjectPhenotypesDS.txt` (981 donantes) |

## Reglas de Particionamiento

Las particiones se construyen combinando dos variables de caracterización:

- **Grupo de tejido** (5 valores): Nervioso, Hematopoyético, Cardiovascular, Musculoesquelético, Visceral/Metabólico  
- **Sexo biológico** (2 valores): Masculino, Femenino

Esto genera **10 particiones** con las siguientes probabilidades de ocurrencia calculadas sobre las 19,788 muestras RNASEQ con sexo conocido:

| # | Grupo de tejido | Sexo | N muestras | Probabilidad |
|---|----------------|------|-----------|-------------|
| P01 | Nervioso | Masculino | 2,838 | 0.1434 |
| P02 | Nervioso | Femenino | 1,066 | 0.0539 |
| P03 | Hematopoyético | Masculino | 932 | 0.0471 |
| P04 | Hematopoyético | Femenino | 475 | 0.0240 |
| P05 | Cardiovascular | Masculino | 1,573 | 0.0795 |
| P06 | Cardiovascular | Femenino | 771 | 0.0390 |
| P07 | Musculoesquelético | Masculino | 2,827 | 0.1429 |
| P08 | Musculoesquelético | Femenino | 1,349 | 0.0682 |
| P09 | Visceral/Metabólico | Masculino | 5,093 | 0.2574 |
| P10 | Visceral/Metabólico | Femenino | 2,864 | 0.1447 |
| | **TOTAL** | | **19,788** | **1.0000** |

### Mapeo de tejidos a grupos

| Grupo | Tejidos GTEx (SMTS) |
|-------|--------------------|
| Nervioso | Brain, Nerve |
| Hematopoyético | Blood, Bone Marrow, Spleen |
| Cardiovascular | Heart, Blood Vessel |
| Musculoesquelético | Muscle, Adipose Tissue, Skin |
| Visceral/Metabólico | Todo lo demás (Liver, Lung, Kidney, Esophagus, Colon, etc.) |

## 1. Imports y configuración de Spark

In [1]:
import sys, os

# point Spark's JVM to the same Python running this notebook
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import FloatType
from functools import reduce

sys.path.insert(0, os.path.abspath('..'))
from GlobalVariables import (
    FILE_PATH, SAMPLE_ATTRS_PATH, SUBJECT_PHENO_PATH,
    OUTPUT_DIR, N_GENES, SAMPLE_STEP, RANDOM_SEED
)

import random
import numpy as np
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f'TPM file:           {FILE_PATH}')
print(f'Sample attributes:  {SAMPLE_ATTRS_PATH}')
print(f'Subject phenotypes: {SUBJECT_PHENO_PATH}')
print(f'Python:             {sys.executable}')

TPM file:           <REPO>/src/data/GTEx_Analysis_2022-06-06_v10_RNASeQCv2.4.2_gene_tpm_non_lcm.gct
Sample attributes:  <REPO>/src/data/GTEx_Analysis_v10_Annotations_SampleAttributesDS.txt
Subject phenotypes: <REPO>/src/data/GTEx_Analysis_v10_Annotations_SubjectPhenotypesDS.txt
Python:             ~/anaconda3/envs/big-data-class/bin/python


In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("GTEx_Partitioning_TC5057") \
    .config("spark.driver.memory", "8g") \
    .getOrCreate()

spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/17 21:09:55 WARN Utils: Your hostname, <HOST>, resolves to a loopback address: 127.0.0.1; using <IP> instead (on interface en0)
26/05/17 21:09:55 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/05/17 21:09:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 2. Carga de metadatos y construcción del índice de particiones

In [3]:
# load sample attributes - keep only RNASEQ samples (the ones present in the TPM file)
sa_df = spark.read.csv(SAMPLE_ATTRS_PATH, sep='\t', header=True) \
    .select('SAMPID', 'SMTS', 'SMAFRZE') \
    .filter(F.col('SMAFRZE') == 'RNASEQ')

# extract subject ID from sample ID: "GTEX-1117F-0005-SM-HL9SH" -> "GTEX-1117F"
sa_df = sa_df.withColumn(
    'SUBJID',
    F.regexp_extract(F.col('SAMPID'), r'^(GTEX-[^-]+)', 1)
)

print(f'Muestras RNASEQ: {sa_df.count():,}')
sa_df.show(5, truncate=False)

Muestras RNASEQ: 19,788
+-----------------------------+-----+-------+----------+
|SAMPID                       |SMTS |SMAFRZE|SUBJID    |
+-----------------------------+-----+-------+----------+
|GTEX-1117F-0005-SM-HL9SH     |Blood|RNASEQ |GTEX-1117F|
|GTEX-1117F-0011-R10b-SM-GI4VE|Brain|RNASEQ |GTEX-1117F|
|GTEX-1117F-0011-R11b-SM-GIN8R|Brain|RNASEQ |GTEX-1117F|
|GTEX-1117F-0011-R2b-SM-GI4VL |Brain|RNASEQ |GTEX-1117F|
|GTEX-1117F-0011-R3a-SM-GJ3PJ |Brain|RNASEQ |GTEX-1117F|
+-----------------------------+-----+-------+----------+
only showing top 5 rows


In [4]:
# load subject phenotypes
sp_df = spark.read.csv(SUBJECT_PHENO_PATH, sep='\t', header=True) \
    .select('SUBJID', 'SEX')

print(f'Donantes: {sp_df.count():,}')
sp_df.show(5)

Donantes: 981
+----------+---+
|    SUBJID|SEX|
+----------+---+
|GTEX-1117F|  2|
|GTEX-111CU|  1|
|GTEX-111FC|  1|
|GTEX-111VG|  1|
|GTEX-111YS|  1|
+----------+---+
only showing top 5 rows


In [5]:
# join to get tissue + sex per sample
meta_df = sa_df.join(sp_df, on='SUBJID', how='inner')

# map raw SMTS values to the 5 tissue groups
tissue_group_col = F.when(F.col('SMTS').isin('Brain', 'Nerve'), 'Nervioso') \
    .when(F.col('SMTS').isin('Blood', 'Bone Marrow', 'Spleen'), 'Hematopoyetico') \
    .when(F.col('SMTS').isin('Heart', 'Blood Vessel'), 'Cardiovascular') \
    .when(F.col('SMTS').isin('Muscle', 'Adipose Tissue', 'Skin'), 'Musculoesqueletico') \
    .otherwise('Visceral_Metabolico')

sex_label_col = F.when(F.col('SEX') == '1', 'Masculino').otherwise('Femenino')

meta_df = meta_df \
    .withColumn('TISSUE_GROUP', tissue_group_col) \
    .withColumn('SEX_LABEL', sex_label_col)

# clean column name for joining: replace hyphens with underscores
meta_df = meta_df.withColumn(
    'COL_NAME',
    F.regexp_replace(F.regexp_replace(F.col('SAMPID'), '-', '_'), '\\.', '_')
)

print(f'Muestras con metadatos completos: {meta_df.count():,}')
meta_df.select('SAMPID', 'SMTS', 'TISSUE_GROUP', 'SEX_LABEL', 'COL_NAME').show(5, truncate=False)

Muestras con metadatos completos: 19,788
+-----------------------------+-----+--------------+---------+-----------------------------+
|SAMPID                       |SMTS |TISSUE_GROUP  |SEX_LABEL|COL_NAME                     |
+-----------------------------+-----+--------------+---------+-----------------------------+
|GTEX-1117F-0005-SM-HL9SH     |Blood|Hematopoyetico|Femenino |GTEX_1117F_0005_SM_HL9SH     |
|GTEX-1117F-0011-R10b-SM-GI4VE|Brain|Nervioso      |Femenino |GTEX_1117F_0011_R10b_SM_GI4VE|
|GTEX-1117F-0011-R11b-SM-GIN8R|Brain|Nervioso      |Femenino |GTEX_1117F_0011_R11b_SM_GIN8R|
|GTEX-1117F-0011-R2b-SM-GI4VL |Brain|Nervioso      |Femenino |GTEX_1117F_0011_R2b_SM_GI4VL |
|GTEX-1117F-0011-R3a-SM-GJ3PJ |Brain|Nervioso      |Femenino |GTEX_1117F_0011_R3a_SM_GJ3PJ |
+-----------------------------+-----+--------------+---------+-----------------------------+
only showing top 5 rows


In [6]:
# verify partition sizes match expected values
print('Tamaño de cada partición:')
meta_df.groupBy('TISSUE_GROUP', 'SEX_LABEL') \
    .count() \
    .orderBy('TISSUE_GROUP', 'SEX_LABEL') \
    .show(20)

Tamaño de cada partición:


+-------------------+---------+-----+
|       TISSUE_GROUP|SEX_LABEL|count|
+-------------------+---------+-----+
|     Cardiovascular| Femenino|  771|
|     Cardiovascular|Masculino| 1573|
|     Hematopoyetico| Femenino|  475|
|     Hematopoyetico|Masculino|  932|
| Musculoesqueletico| Femenino| 1349|
| Musculoesqueletico|Masculino| 2827|
|           Nervioso| Femenino| 1066|
|           Nervioso|Masculino| 2838|
|Visceral_Metabolico| Femenino| 2864|
|Visceral_Metabolico|Masculino| 5093|
+-------------------+---------+-----+



## 3. Carga del dataset TPM

Se carga el archivo GCT principal. Para mantener tiempos razonables en esta etapa de prueba, se toma 1 de cada `SAMPLE_STEP` columnas. En la etapa de entrenamiento/prueba se usarán todas las columnas.

In [7]:
import pandas as pd
from pyspark.sql.functions import split as spark_split

# read column names with pandas (only header row — fast)
peek = pd.read_csv(FILE_PATH, sep='\t', skiprows=2, nrows=0)
all_col_names = peek.columns.tolist()

# sample every SAMPLE_STEP-th column
selected_indices = [0, 1] + list(range(2, len(all_col_names), SAMPLE_STEP))
selected_cols_raw = [all_col_names[i] for i in selected_indices]
clean_names = [c.replace('-', '_').replace('.', '_') for c in selected_cols_raw]
sample_cols = clean_names[2:]

print(f'Columnas de muestra seleccionadas: {len(sample_cols)} de {len(all_col_names) - 2:,}')

# use spark.read.text + pure Spark SQL to avoid Python RDD lambdas (Windows compatibility)
raw_df = spark.read.text(FILE_PATH)

# filter to gene rows using Spark column expression — no Python lambda
gene_df = raw_df.filter(F.col('value').startswith('ENSG'))

# split each line on tab and pick selected columns by index
split_col = spark_split(F.col('value'), '\t')
df_tpm = gene_df.select(
    *[split_col.getItem(i).alias(clean_names[idx]) for idx, i in enumerate(selected_indices)]
)

# cast sample columns to float
for c in sample_cols:
    df_tpm = df_tpm.withColumn(c, F.col(c).cast(FloatType()))

print(f'DataFrame TPM cargado: {df_tpm.count():,} genes x {len(clean_names)} columnas')
df_tpm.select(clean_names[:5]).show(3)

Columnas de muestra seleccionadas: 197 de 19,616


DataFrame TPM cargado: 59,033 genes x 199 columnas
+-----------------+-----------+------------------------+------------------------+----------------------------+
|             Name|Description|GTEX_1117F_0005_SM_HL9SH|GTEX_111YS_1426_SM_5GID8|GTEX_117YW_0011_R8a_SM_GINAB|
+-----------------+-----------+------------------------+------------------------+----------------------------+
|ENSG00000223972.5|    DDX11L1|                     0.0|                     0.0|                         0.0|
|ENSG00000227232.5|     WASH7P|                 1.33343|                 1.57221|                     3.12039|
|ENSG00000278267.1|  MIR6859-1|                     0.0|                     0.0|                         0.0|
+-----------------+-----------+------------------------+------------------------+----------------------------+
only showing top 3 rows


## 4. Función auxiliar de particionamiento

Dado un grupo de tejido y un sexo, devuelve el subconjunto de columnas del DataFrame TPM que pertenecen a esa partición, junto con `Name` y `Description`.

In [8]:
# pre-build a dict: col_name -> (tissue_group, sex_label) for fast lookup
meta_lookup = {
    row['COL_NAME']: (row['TISSUE_GROUP'], row['SEX_LABEL'])
    for row in meta_df.select('COL_NAME', 'TISSUE_GROUP', 'SEX_LABEL').collect()
}

def get_partition_cols(tissue_group: str, sex_label: str) -> list:
    """Return column names in df_tpm that belong to the given partition."""
    return [
        c for c in sample_cols
        if meta_lookup.get(c) == (tissue_group, sex_label)
    ]

def get_partition(tissue_group: str, sex_label: str):
    """Return a DataFrame with Name, Description, and only the columns for this partition."""
    cols = get_partition_cols(tissue_group, sex_label)
    if not cols:
        print(f'[WARN] No hay columnas muestreadas para {tissue_group} + {sex_label}')
        return None
    return df_tpm.select(['Name', 'Description'] + cols)

print('Columnas disponibles por partición (en la muestra 1/SAMPLE_STEP):')
partitions = [
    ('Nervioso',            'Masculino'),
    ('Nervioso',            'Femenino'),
    ('Hematopoyetico',      'Masculino'),
    ('Hematopoyetico',      'Femenino'),
    ('Cardiovascular',      'Masculino'),
    ('Cardiovascular',      'Femenino'),
    ('Musculoesqueletico',  'Masculino'),
    ('Musculoesqueletico',  'Femenino'),
    ('Visceral_Metabolico', 'Masculino'),
    ('Visceral_Metabolico', 'Femenino'),
]
for tg, sx in partitions:
    n = len(get_partition_cols(tg, sx))
    print(f'  {tg:<25} + {sx:<12} -> {n} columnas')

Columnas disponibles por partición (en la muestra 1/SAMPLE_STEP):
  Nervioso                  + Masculino    -> 30 columnas
  Nervioso                  + Femenino     -> 8 columnas
  Hematopoyetico            + Masculino    -> 10 columnas
  Hematopoyetico            + Femenino     -> 6 columnas
  Cardiovascular            + Masculino    -> 14 columnas
  Cardiovascular            + Femenino     -> 7 columnas
  Musculoesqueletico        + Masculino    -> 29 columnas
  Musculoesqueletico        + Femenino     -> 9 columnas
  Visceral_Metabolico       + Masculino    -> 60 columnas
  Visceral_Metabolico       + Femenino     -> 24 columnas


## 5. Extracción de muestras por partición

Para cada una de las 10 particiones se muestra: dimensiones del subconjunto, los primeros genes y estadísticas básicas de TPM. **Estas son muestras de prueba** para verificar el funcionamiento del código; en la etapa posterior se usarán todas las columnas.

### P01 — Nervioso + Masculino

In [9]:
p01 = get_partition('Nervioso', 'Masculino')
cols_p01 = get_partition_cols('Nervioso', 'Masculino')
print(f'P01 — Nervioso + Masculino: {N_GENES:,} genes x {len(cols_p01)} muestras (muestra 1/{SAMPLE_STEP})')
p01.select(['Name', 'Description'] + cols_p01[:3]).show(5, truncate=True)
p01.select(cols_p01[:3]).describe().show()

P01 — Nervioso + Masculino: 59,033 genes x 30 muestras (muestra 1/100)
+-----------------+-----------+----------------------------+------------------------+------------------------+
|             Name|Description|GTEX_117YW_0011_R8a_SM_GINAB|GTEX_12ZZW_2926_SM_5LZUP|GTEX_131XH_2526_SM_5GCND|
+-----------------+-----------+----------------------------+------------------------+------------------------+
|ENSG00000223972.5|    DDX11L1|                         0.0|                     0.0|                0.038963|
|ENSG00000227232.5|     WASH7P|                     3.12039|                 1.92342|                 16.8925|
|ENSG00000278267.1|  MIR6859-1|                         0.0|                     0.0|                     0.0|
|ENSG00000243485.5|MIR1302-2HG|                         0.0|                0.034297|                     0.0|
|ENSG00000237613.2|    FAM138A|                         0.0|                     0.0|                     0.0|
+-----------------+-----------+----------

+-------+----------------------------+------------------------+------------------------+
|summary|GTEX_117YW_0011_R8a_SM_GINAB|GTEX_12ZZW_2926_SM_5LZUP|GTEX_131XH_2526_SM_5GCND|
+-------+----------------------------+------------------------+------------------------+
|  count|                       59033|                   59033|                   59033|
|   mean|           16.93967892698064|      16.939679941761224|       16.93967882172353|
| stddev|           679.0741427116366|       748.4873006848754|       281.1508770489939|
|    min|                         0.0|                     0.0|                     0.0|
|    max|                     74399.5|                 74510.6|                 33489.1|
+-------+----------------------------+------------------------+------------------------+



### P02 — Nervioso + Femenino

In [10]:
p02 = get_partition('Nervioso', 'Femenino')
cols_p02 = get_partition_cols('Nervioso', 'Femenino')
print(f'P02 — Nervioso + Femenino: {N_GENES:,} genes x {len(cols_p02)} muestras')
p02.select(['Name', 'Description'] + cols_p02[:3]).show(5, truncate=True)
p02.select(cols_p02[:3]).describe().show()

P02 — Nervioso + Femenino: 59,033 genes x 8 muestras
+-----------------+-----------+------------------------+----------------------------+------------------------+
|             Name|Description|GTEX_13O3O_3126_SM_5KM3H|GTEX_15DCD_0011_R8b_SM_7KUEF|GTEX_18D9A_1726_SM_731BS|
+-----------------+-----------+------------------------+----------------------------+------------------------+
|ENSG00000223972.5|    DDX11L1|                     0.0|                    0.019567|                     0.0|
|ENSG00000227232.5|     WASH7P|                 2.06995|                     1.40112|                 9.03411|
|ENSG00000278267.1|  MIR6859-1|                     0.0|                         0.0|                     0.0|
|ENSG00000243485.5|MIR1302-2HG|                     0.0|                    0.078131|                     0.0|
|ENSG00000237613.2|    FAM138A|                     0.0|                         0.0|                     0.0|
+-----------------+-----------+------------------------+---

+-------+------------------------+----------------------------+------------------------+
|summary|GTEX_13O3O_3126_SM_5KM3H|GTEX_15DCD_0011_R8b_SM_7KUEF|GTEX_18D9A_1726_SM_731BS|
+-------+------------------------+----------------------------+------------------------+
|  count|                   59033|                       59033|                   59033|
|   mean|      16.939680595855375|          16.939676625238622|      16.939677078117125|
| stddev|       727.6934034967661|           721.3952489058491|      226.14058997086505|
|    min|                     0.0|                         0.0|                     0.0|
|    max|                 69370.1|                     92149.8|                 25625.2|
+-------+------------------------+----------------------------+------------------------+



### P03 — Hematopoyético + Masculino

In [11]:
p03 = get_partition('Hematopoyetico', 'Masculino')
cols_p03 = get_partition_cols('Hematopoyetico', 'Masculino')
print(f'P03 — Hematopoyético + Masculino: {N_GENES:,} genes x {len(cols_p03)} muestras')
p03.select(['Name', 'Description'] + cols_p03[:3]).show(5, truncate=True)
p03.select(cols_p03[:3]).describe().show()

P03 — Hematopoyético + Masculino: 59,033 genes x 10 muestras
+-----------------+-----------+------------------------+------------------------+------------------------+
|             Name|Description|GTEX_12WSA_0005_SM_5O9BM|GTEX_13JVG_0005_SM_HKZYO|GTEX_14ICK_0006_SM_5NQB5|
+-----------------+-----------+------------------------+------------------------+------------------------+
|ENSG00000223972.5|    DDX11L1|                0.019888|                0.054483|                     0.0|
|ENSG00000227232.5|     WASH7P|                 3.46975|                 1.48962|                 3.02711|
|ENSG00000278267.1|  MIR6859-1|                     0.0|                     0.0|                     0.0|
|ENSG00000243485.5|MIR1302-2HG|                0.079415|                     0.0|                     0.0|
|ENSG00000237613.2|    FAM138A|                     0.0|                     0.0|                0.065303|
+-----------------+-----------+------------------------+------------------------+--

+-------+------------------------+------------------------+------------------------+
|summary|GTEX_12WSA_0005_SM_5O9BM|GTEX_13JVG_0005_SM_HKZYO|GTEX_14ICK_0006_SM_5NQB5|
+-------+------------------------+------------------------+------------------------+
|  count|                   59033|                   59033|                   59033|
|   mean|      16.939670789872345|      16.939668482874875|       16.93966941506421|
| stddev|       861.3024769409301|      1709.0382354658361|       575.8775725153226|
|    min|                     0.0|                     0.0|                     0.0|
|    max|                156775.0|                361793.0|                123505.0|
+-------+------------------------+------------------------+------------------------+



### P04 — Hematopoyético + Femenino

In [12]:
p04 = get_partition('Hematopoyetico', 'Femenino')
cols_p04 = get_partition_cols('Hematopoyetico', 'Femenino')
print(f'P04 — Hematopoyético + Femenino: {N_GENES:,} genes x {len(cols_p04)} muestras')
p04.select(['Name', 'Description'] + cols_p04[:3]).show(5, truncate=True)
p04.select(cols_p04[:3]).describe().show()

P04 — Hematopoyético + Femenino: 59,033 genes x 6 muestras
+-----------------+-----------+------------------------+------------------------+-----------------------+
|             Name|Description|GTEX_1117F_0005_SM_HL9SH|GTEX_17JCI_0005_SM_7P8OL|GTEX_T5JW_0003_SM_3NMAD|
+-----------------+-----------+------------------------+------------------------+-----------------------+
|ENSG00000223972.5|    DDX11L1|                     0.0|                0.044362|                    0.0|
|ENSG00000227232.5|     WASH7P|                 1.33343|                 1.09738|                3.07818|
|ENSG00000278267.1|  MIR6859-1|                     0.0|                     0.0|                    0.0|
|ENSG00000243485.5|MIR1302-2HG|                     0.0|                0.044285|                    0.0|
|ENSG00000237613.2|    FAM138A|                     0.0|                     0.0|                    0.0|
+-----------------+-----------+------------------------+------------------------+------------

+-------+------------------------+------------------------+-----------------------+
|summary|GTEX_1117F_0005_SM_HL9SH|GTEX_17JCI_0005_SM_7P8OL|GTEX_T5JW_0003_SM_3NMAD|
+-------+------------------------+------------------------+-----------------------+
|  count|                   59033|                   59033|                  59033|
|   mean|      16.939676249088276|      16.939673090420158|     16.939676556368397|
| stddev|      2826.5371859700417|       899.4244559877137|     255.77373250964115|
|    min|                     0.0|                     0.0|                    0.0|
|    max|                674027.0|                182852.0|                22636.4|
+-------+------------------------+------------------------+-----------------------+



### P05 — Cardiovascular + Masculino

In [13]:
p05 = get_partition('Cardiovascular', 'Masculino')
cols_p05 = get_partition_cols('Cardiovascular', 'Masculino')
print(f'P05 — Cardiovascular + Masculino: {N_GENES:,} genes x {len(cols_p05)} muestras')
p05.select(['Name', 'Description'] + cols_p05[:3]).show(5, truncate=True)
p05.select(cols_p05[:3]).describe().show()

P05 — Cardiovascular + Masculino: 59,033 genes x 14 muestras
+-----------------+-----------+------------------------+------------------------+------------------------+
|             Name|Description|GTEX_11TT1_1526_SM_5EQKU|GTEX_12WSE_0926_SM_5S2VX|GTEX_13FHP_0826_SM_5K7V5|
+-----------------+-----------+------------------------+------------------------+------------------------+
|ENSG00000223972.5|    DDX11L1|                0.026111|                0.033218|                0.016764|
|ENSG00000227232.5|     WASH7P|                 1.80173|                 2.22733|                 3.62308|
|ENSG00000278267.1|  MIR6859-1|                     0.0|                     0.0|                     0.0|
|ENSG00000243485.5|MIR1302-2HG|                0.052131|                     0.0|                     0.0|
|ENSG00000237613.2|    FAM138A|                     0.0|                0.023558|                0.023777|
+-----------------+-----------+------------------------+------------------------+--

+-------+------------------------+------------------------+------------------------+
|summary|GTEX_11TT1_1526_SM_5EQKU|GTEX_12WSE_0926_SM_5S2VX|GTEX_13FHP_0826_SM_5K7V5|
+-------+------------------------+------------------------+------------------------+
|  count|                   59033|                   59033|                   59033|
|   mean|       16.93967537946057|      16.939676226222215|      16.939677577406908|
| stddev|      318.36882006609875|       747.0845102026288|       619.7409213958288|
|    min|                     0.0|                     0.0|                     0.0|
|    max|                 31737.2|                 78147.2|                 63660.1|
+-------+------------------------+------------------------+------------------------+



### P06 — Cardiovascular + Femenino

In [14]:
p06 = get_partition('Cardiovascular', 'Femenino')
cols_p06 = get_partition_cols('Cardiovascular', 'Femenino')
print(f'P06 — Cardiovascular + Femenino: {N_GENES:,} genes x {len(cols_p06)} muestras')
p06.select(['Name', 'Description'] + cols_p06[:3]).show(5, truncate=True)
p06.select(cols_p06[:3]).describe().show()

P06 — Cardiovascular + Femenino: 59,033 genes x 7 muestras
+-----------------+-----------+------------------------+------------------------+------------------------+
|             Name|Description|GTEX_11EMC_0826_SM_59862|GTEX_12WSK_0426_SM_5GCNS|GTEX_15CHC_0326_SM_5ZZVP|
+-----------------+-----------+------------------------+------------------------+------------------------+
|ENSG00000223972.5|    DDX11L1|                0.015234|                0.037847|                     0.0|
|ENSG00000227232.5|     WASH7P|                 1.64619|                 4.58254|                 1.33702|
|ENSG00000278267.1|  MIR6859-1|                     0.0|                     0.0|                     0.0|
|ENSG00000243485.5|MIR1302-2HG|                0.030415|                     0.0|                     0.0|
|ENSG00000237613.2|    FAM138A|                0.043214|                     0.0|                     0.0|
+-----------------+-----------+------------------------+------------------------+----

+-------+------------------------+------------------------+------------------------+
|summary|GTEX_11EMC_0826_SM_59862|GTEX_12WSK_0426_SM_5GCNS|GTEX_15CHC_0326_SM_5ZZVP|
+-------+------------------------+------------------------+------------------------+
|  count|                   59033|                   59033|                   59033|
|   mean|      16.939679366092687|      16.939677693982098|      16.939676948491805|
| stddev|       816.2527646430032|      227.83574906917113|        710.485342919841|
|    min|                     0.0|                     0.0|                     0.0|
|    max|                 96915.5|                 17339.7|                 83175.0|
+-------+------------------------+------------------------+------------------------+



### P07 — Musculoesquelético + Masculino

In [15]:
p07 = get_partition('Musculoesqueletico', 'Masculino')
cols_p07 = get_partition_cols('Musculoesqueletico', 'Masculino')
print(f'P07 — Musculoesquelético + Masculino: {N_GENES:,} genes x {len(cols_p07)} muestras')
p07.select(['Name', 'Description'] + cols_p07[:3]).show(5, truncate=True)
p07.select(cols_p07[:3]).describe().show()

P07 — Musculoesquelético + Masculino: 59,033 genes x 29 muestras
+-----------------+-----------+------------------------+------------------------+------------------------+
|             Name|Description|GTEX_11LCK_1226_SM_5Q5AM|GTEX_11O72_0226_SM_59869|GTEX_1314G_1526_SM_5EGK2|
+-----------------+-----------+------------------------+------------------------+------------------------+
|ENSG00000223972.5|    DDX11L1|                     0.0|                     0.0|                     0.0|
|ENSG00000227232.5|     WASH7P|                0.633564|                 4.43226|                 4.24087|
|ENSG00000278267.1|  MIR6859-1|                     0.0|                     0.0|                     0.0|
|ENSG00000243485.5|MIR1302-2HG|                0.084484|                     0.0|                     0.0|
|ENSG00000237613.2|    FAM138A|                     0.0|                     0.0|                     0.0|
+-----------------+-----------+------------------------+-----------------------

+-------+------------------------+------------------------+------------------------+
|summary|GTEX_11LCK_1226_SM_5Q5AM|GTEX_11O72_0226_SM_59869|GTEX_1314G_1526_SM_5EGK2|
+-------+------------------------+------------------------+------------------------+
|  count|                   59033|                   59033|                   59033|
|   mean|      16.939675208859686|      16.939675892894776|      16.939677506866218|
| stddev|       653.2951960736632|       512.6221333375215|      326.67193592141746|
|    min|                     0.0|                     0.0|                     0.0|
|    max|                 71912.1|                 57165.7|                 31782.2|
+-------+------------------------+------------------------+------------------------+



### P08 — Musculoesquelético + Femenino

In [16]:
p08 = get_partition('Musculoesqueletico', 'Femenino')
cols_p08 = get_partition_cols('Musculoesqueletico', 'Femenino')
print(f'P08 — Musculoesquelético + Femenino: {N_GENES:,} genes x {len(cols_p08)} muestras')
p08.select(['Name', 'Description'] + cols_p08[:3]).show(5, truncate=True)
p08.select(cols_p08[:3]).describe().show()

P08 — Musculoesquelético + Femenino: 59,033 genes x 9 muestras
+-----------------+-----------+------------------------+------------------------+------------------------+
|             Name|Description|GTEX_139D8_0726_SM_5P9GJ|GTEX_146FH_2026_SM_5MR6J|GTEX_1H2FU_0526_SM_ACKXO|
+-----------------+-----------+------------------------+------------------------+------------------------+
|ENSG00000223972.5|    DDX11L1|                     0.0|                     0.0|                0.014971|
|ENSG00000227232.5|     WASH7P|                  4.5945|                 1.97964|                 1.81272|
|ENSG00000278267.1|  MIR6859-1|                     0.0|                     0.0|                     0.0|
|ENSG00000243485.5|MIR1302-2HG|                     0.0|                     0.0|                     0.0|
|ENSG00000237613.2|    FAM138A|                     0.0|                0.027649|                0.021235|
+-----------------+-----------+------------------------+------------------------+

+-------+------------------------+------------------------+------------------------+
|summary|GTEX_139D8_0726_SM_5P9GJ|GTEX_146FH_2026_SM_5MR6J|GTEX_1H2FU_0526_SM_ACKXO|
+-------+------------------------+------------------------+------------------------+
|  count|                   59033|                   59033|                   59033|
|   mean|       16.93967804830544|       16.93967903729178|       16.93967715290919|
| stddev|      437.39468216252794|       312.4479935772466|       426.2819534965633|
|    min|                     0.0|                     0.0|                     0.0|
|    max|                 40523.6|                 30592.6|                 40802.5|
+-------+------------------------+------------------------+------------------------+



### P09 — Visceral/Metabólico + Masculino

In [17]:
p09 = get_partition('Visceral_Metabolico', 'Masculino')
cols_p09 = get_partition_cols('Visceral_Metabolico', 'Masculino')
print(f'P09 — Visceral/Metabólico + Masculino: {N_GENES:,} genes x {len(cols_p09)} muestras')
p09.select(['Name', 'Description'] + cols_p09[:3]).show(5, truncate=True)
p09.select(cols_p09[:3]).describe().show()

P09 — Visceral/Metabólico + Masculino: 59,033 genes x 60 muestras
+-----------------+-----------+------------------------+------------------------+------------------------+
|             Name|Description|GTEX_111YS_1426_SM_5GID8|GTEX_11DXW_1226_SM_5H133|GTEX_11DYG_0826_SM_5N9GH|
+-----------------+-----------+------------------------+------------------------+------------------------+
|ENSG00000223972.5|    DDX11L1|                     0.0|                0.026297|                     0.0|
|ENSG00000227232.5|     WASH7P|                 1.57221|                 4.86166|                 6.57587|
|ENSG00000278267.1|  MIR6859-1|                     0.0|                     0.0|                     0.0|
|ENSG00000243485.5|MIR1302-2HG|                     0.0|                0.052502|                0.051979|
|ENSG00000237613.2|    FAM138A|                     0.0|                     0.0|                     0.0|
+-----------------+-----------+------------------------+----------------------

+-------+------------------------+------------------------+------------------------+
|summary|GTEX_111YS_1426_SM_5GID8|GTEX_11DXW_1226_SM_5H133|GTEX_11DYG_0826_SM_5N9GH|
+-------+------------------------+------------------------+------------------------+
|  count|                   59033|                   59033|                   59033|
|   mean|       16.93967929867567|       16.93967350765183|       16.93967813474517|
| stddev|       706.0269008472371|       621.6241322027068|      184.45856837374794|
|    min|                     0.0|                     0.0|                     0.0|
|    max|                 84388.2|                126338.0|                 23813.9|
+-------+------------------------+------------------------+------------------------+



### P10 — Visceral/Metabólico + Femenino

In [18]:
p10 = get_partition('Visceral_Metabolico', 'Femenino')
cols_p10 = get_partition_cols('Visceral_Metabolico', 'Femenino')
print(f'P10 — Visceral/Metabólico + Femenino: {N_GENES:,} genes x {len(cols_p10)} muestras')
p10.select(['Name', 'Description'] + cols_p10[:3]).show(5, truncate=True)
p10.select(cols_p10[:3]).describe().show()

P10 — Visceral/Metabólico + Femenino: 59,033 genes x 24 muestras
+-----------------+-----------+------------------------+------------------------+------------------------+
|             Name|Description|GTEX_11VI4_0726_SM_5GU5B|GTEX_11ZTS_1926_SM_5CVLA|GTEX_1211K_0126_SM_59HJE|
+-----------------+-----------+------------------------+------------------------+------------------------+
|ENSG00000223972.5|    DDX11L1|                0.033816|                0.034198|                0.016271|
|ENSG00000227232.5|     WASH7P|                 4.49079|                 3.20574|                 1.14396|
|ENSG00000278267.1|  MIR6859-1|                     0.0|                     0.0|                     0.0|
|ENSG00000243485.5|MIR1302-2HG|                     0.0|                     0.0|                0.032486|
|ENSG00000237613.2|    FAM138A|                     0.0|                     0.0|                     0.0|
+-----------------+-----------+------------------------+-----------------------

+-------+------------------------+------------------------+------------------------+
|summary|GTEX_11VI4_0726_SM_5GU5B|GTEX_11ZTS_1926_SM_5CVLA|GTEX_1211K_0126_SM_59HJE|
+-------+------------------------+------------------------+------------------------+
|  count|                   59033|                   59033|                   59033|
|   mean|       16.93967578983757|      16.939677485478928|      16.939679208178834|
| stddev|      334.41565346254384|      189.82843852256582|       599.1276473923651|
|    min|                     0.0|                     0.0|                     0.0|
|    max|                 34022.2|                 13266.6|                 58721.4|
+-------+------------------------+------------------------+------------------------+



## 6. Resumen de particiones

Tabla de verificación final con el conteo de columnas disponibles en la muestra 1/SAMPLE_STEP para cada partición.

In [19]:
import pandas as pd

summary_rows = []
full_counts = {
    ('Nervioso',            'Masculino'): 2838,
    ('Nervioso',            'Femenino'):  1066,
    ('Hematopoyetico',      'Masculino'): 932,
    ('Hematopoyetico',      'Femenino'):  475,
    ('Cardiovascular',      'Masculino'): 1573,
    ('Cardiovascular',      'Femenino'):  771,
    ('Musculoesqueletico',  'Masculino'): 2827,
    ('Musculoesqueletico',  'Femenino'):  1349,
    ('Visceral_Metabolico', 'Masculino'): 5093,
    ('Visceral_Metabolico', 'Femenino'):  2864,
}
total = 19788

for i, (tg, sx) in enumerate(partitions, 1):
    n_full = full_counts[(tg, sx)]
    n_sampled = len(get_partition_cols(tg, sx))
    summary_rows.append({
        'Partición': f'P{i:02d}',
        'Grupo tejido': tg,
        'Sexo': sx,
        'N total': n_full,
        'Prob': round(n_full / total, 4),
        f'N muestra (1/{SAMPLE_STEP})': n_sampled,
    })

pd.DataFrame(summary_rows)

,Partición,Grupo tejido,Sexo,N total,Prob,N muestra (1/100)
0,P01,Nervioso,Masculino,2838,0.1434,30
1,P02,Nervioso,Femenino,1066,0.0539,8
2,P03,Hematopoyetico,Masculino,932,0.0471,10
3,P04,Hematopoyetico,Femenino,475,0.0240,6
4,P05,Cardiovascular,Masculino,1573,0.0795,14
5,P06,Cardiovascular,Femenino,771,0.0390,7
6,P07,Musculoesqueletico,Masculino,2827,0.1429,29
7,P08,Musculoesqueletico,Femenino,1349,0.0682,9
8,P09,Visceral_Metabolico,Masculino,5093,0.2574,60
9,P10,Visceral_Metabolico,Femenino,2864,0.1447,24


## Referencias

1. GTEx Consortium. (2020). The GTEx Consortium atlas of genetic regulatory effects across human tissues. *Science*. https://doi.org/10.1126/science.aaz1776
2. GTEx Portal. (2025). GTEx Analysis V10 Downloads. Broad Institute. https://gtexportal.org/home/downloads/adult-gtex

## Declaración de uso de Inteligencia Artificial

Anthropic. (2026). *Claude Sonnet 4.6* [Modelo de lenguaje grande], utilizado para soporte en documentación, redacción de celdas markdown y revisión de estructura del notebook. https://claude.ai

*La responsabilidad final sobre el contenido entregado recae en los autores del equipo. Las soluciones de código, la selección de variables, el diseño del particionamiento y las decisiones analíticas son propias del equipo.*